In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests
import time, json, base64, re
from transformers import pipeline

import warnings

warnings.filterwarnings("ignore")

pipe = pipeline(
    "automatic-speech-recognition", model=r"D:\Developers\Kaustubh\whisper-medium"
)

# Chrome options
options = webdriver.ChromeOptions()

# Enable network/performance logging
options.set_capability("goog:loggingPrefs", {"performance": "ALL"})


headers = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json;charset=UTF-8",
    "Origin": "https://services.gst.gov.in",
    "Referer": "https://services.gst.gov.in/services/searchtpbypan",
}


pans = [
    "AAACI4798L",
    "AAACG1653N",
    "AAACH0351E",
    "AAACV7244E",
    "AADCS1718H",
    "AAACI4341M",
    "AAACH1458C",
    "AAACR4849R",
    "AACCT8243P",
    "AACCA1963B",
]

driver = webdriver.Chrome(options=options)

api_url = r"https://services.gst.gov.in/services/api/get/gstndtls"

# Enable DevTools Network
driver.execute_cdp_cmd("Network.enable", {})
try:

    driver.get("https://services.gst.gov.in/services/searchtpbypan")
    wait = WebDriverWait(driver, 20)
    wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, ".dimmer-holder")))
    textbox = wait.until(EC.presence_of_element_located((By.ID, "for_gstin")))

    for i in pans:
        textbox.clear()
        textbox.send_keys(f"{i}")

        # Click speaker button
        audio_button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[i[contains(@class,'fa-volume-up')]]")
            )
        )

        audio_button.click()

        print("Audio button clicked...")

        time.sleep(5)

        logs = driver.get_log("performance")

        request_id = None

        for entry in logs:

            try:
                msg = json.loads(entry["message"])["message"]

                if msg["method"] == "Network.responseReceived":
                    url = msg["params"]["response"]["url"]
                    if "audiocaptcha" in url:

                        print("Found audio request:")
                        print(url)
                        request_id = msg["params"]["requestId"]

                        break

            except Exception:
                pass

        if not request_id:
            raise Exception("Could not find audiocaptcha request in network logs.")

        # Get response body
        body = driver.execute_cdp_cmd(
            "Network.getResponseBody", {"requestId": request_id}
        )

        if body.get("base64Encoded"):

            audio_bytes = base64.b64decode(body["body"])

        else:

            audio_bytes = body["body"].encode()

        # Save audio
        audio_path = f"captcha_audio_{i}.wav"

        with open(audio_path, "wb") as f:
            f.write(audio_bytes)

        print("Saved: captcha_audio.wav")
        result = pipe(audio_path)

        result = re.sub(r"[^0-9]", "", result["text"])
        print(f"DETECTED AUDIO: {result}")

        import requests

        session = requests.Session()

        for cookie in driver.get_cookies():
            session.cookies.set(cookie["name"], cookie["value"])

        print(session.cookies.get_dict())

        for _pan_ in pans:
            payload = {"panNO": i, "captcha": result}

            response = session.post(api_url, json=payload, headers=headers, verify=False)


            if response.ok:
                print("SUCCESS !!!")
                data = response.json()
                print(response.headers.get("Content-Type"))
                print(data)

        enter_c = wait.until(EC.presence_of_element_located((By.ID, "for_gstin")))

        enter_c.clear()
        enter_c.send_keys(str(result))
        search_button = wait.until(EC.element_to_be_clickable((By.ID, "lotsearch")))
        search_button.click()

        refresh_button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[i[contains(@class,'fa-refresh')]]")
            )
        )

        refresh_button.click()

except Exception as e:
    print("ERROR:", e)

finally:

    driver.quit()


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import time, json, base64, re
from transformers import pipeline

import warnings

warnings.filterwarnings("ignore")

pipe = pipeline(
    "automatic-speech-recognition", model=r"D:\Developers\Kaustubh\whisper-medium"
)

# Chrome options
options = webdriver.ChromeOptions()

# Enable network/performance logging
options.set_capability("goog:loggingPrefs", {"performance": "ALL"})


headers = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json;charset=UTF-8",
    "Origin": "https://services.gst.gov.in",
    "Referer": "https://services.gst.gov.in/services/searchtpbypan",
}


pans = [
    "AAACI4798L",
    "AAACG1653N",
    "AAACH0351E",
    "AAACV7244E",
    "AADCS1718H",
    "AAACI4341M",
    "AAACH1458C",
    "AAACR4849R",
    "AACCT8243P",
    "AACCA1963B",
]

driver = webdriver.Chrome(options=options)

api_url = r"https://services.gst.gov.in/services/api/get/gstndtls"

# Enable DevTools Network
driver.execute_cdp_cmd("Network.enable", {})
try:

    driver.get("https://services.gst.gov.in/services/searchtpbypan")
    wait = WebDriverWait(driver, 20)
    wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, ".dimmer-holder")))
    textbox = wait.until(EC.presence_of_element_located((By.ID, "for_gstin")))
    textbox.clear()
    textbox.send_keys(f"{i}")

    # Click speaker button
    audio_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[i[contains(@class,'fa-volume-up')]]")
        )
    )

    audio_button.click()

    print("Audio button clicked...")

    time.sleep(5)

    logs = driver.get_log("performance")

    request_id = None

    for entry in logs:

        try:
            msg = json.loads(entry["message"])["message"]

            if msg["method"] == "Network.responseReceived":
                url = msg["params"]["response"]["url"]
                if "audiocaptcha" in url:

                    print("Found audio request:")
                    print(url)
                    request_id = msg["params"]["requestId"]

                    break

        except Exception:
            pass

    if not request_id:
        raise Exception("Could not find audiocaptcha request in network logs.")

    # Get response body
    body = driver.execute_cdp_cmd(
        "Network.getResponseBody", {"requestId": request_id}
    )

    if body.get("base64Encoded"):

        audio_bytes = base64.b64decode(body["body"])

    else:

        audio_bytes = body["body"].encode()

    # Save audio
    audio_path = f"captcha_audio_{i}.wav"

    with open(audio_path, "wb") as f:
        f.write(audio_bytes)

    print("Saved: captcha_audio.wav")
    result = pipe(audio_path)

    result = re.sub(r"[^0-9]", "", result["text"])
    print(f"DETECTED AUDIO: {result}")



    session = requests.Session()

    for cookie in driver.get_cookies():
        session.cookies.set(cookie["name"], cookie["value"])

    print(session.cookies.get_dict())

    headers = {
        "Accept": "application/json, text/plain, */*",
        "Content-Type": "application/json;charset=UTF-8",
        "Origin": "https://services.gst.gov.in",
        "Referer": "https://services.gst.gov.in/services/searchtpbypan",
    }

    payload = {"panNO": i, "captcha": result}

    response = session.post(api_url, json=payload, headers=headers, verify=False)

    print("Status:", response.status_code)
    # print("Response:")
    # print(response.text)

    if response.ok:
        print("SUCCESS !!!")
        data = response.json()
        print(response.headers.get("Content-Type"))
        print(data)

    enter_c = wait.until(EC.presence_of_element_located((By.ID, "for_gstin")))

    enter_c.clear()
    enter_c.send_keys(str(result))
    search_button = wait.until(EC.element_to_be_clickable((By.ID, "lotsearch")))
    search_button.click()

    refresh_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[i[contains(@class,'fa-refresh')]]")
        )
    )

    refresh_button.click()

except Exception as e:
    print("ERROR:", e)

finally:

    driver.quit()


In [ ]:

# https://services.gst.gov.in/services/api/get/gstndtls
# {"panNO":"AAACI4798L","captcha":"327933"}

In [5]:
state_map = {
    "35": "Andaman and Nicobar Islands",
    "37": "Andhra Pradesh",
    "12": "Arunachal Pradesh",
    "18": "Assam",
    "10": "Bihar",
    "99": "CBIC",
    "04": "Chandigarh",
    "22": "Chhattisgarh",
    "26": "Dadra and Nagar Haveli and Daman and Diu",
    "25": "Daman and Diu",
    "07": "Delhi",
    "30": "Goa",
    "24": "Gujarat",
    "06": "Haryana",
    "02": "Himachal Pradesh",
    "01": "Jammu and Kashmir",
    "20": "Jharkhand",
    "29": "Karnataka",
    "32": "Kerala",
    "38": "Ladakh",
    "31": "Lakshadweep",
    "23": "Madhya Pradesh",
    "27": "Maharashtra",
    "14": "Manipur",
    "17": "Meghalaya",
    "15": "Mizoram",
    "13": "Nagaland",
    "21": "Odisha",
    "97": "Other Territory",
    "34": "Puducherry",
    "03": "Punjab",
    "08": "Rajasthan",
    "11": "Sikkim",
    "33": "Tamil Nadu",
    "36": "Telangana",
    "16": "Tripura",
    "09": "Uttar Pradesh",
    "05": "Uttarakhand",
    "19": "West Bengal"
}


dir = r"C:\Users\kaustubh.keny\Downloads\gst_json_results"

from pathlib import Path
import json
from utils import Helper
import pandas as pd
from datetime import datetime
utils = Helper()

json_dir = Path(dir)
json_data = json_dir.glob("*.json")
final_data = []

for jdir in json_data:
    
    data = utils.load_json(jdir)
    dataList = data["gstinResList"]
    for content in dataList:
        content["pan"] = jdir.stem
    
    final_data.extend(dataList)


df = pd.DataFrame(final_data)

df["state"] = df.stateCd.apply( lambda x: state_map[str(x)] if x else "NOT FOUND")
df.to_excel(f"FINAL_PAN_DATA_{datetime.now().strftime("%d%m%Y%H%M")}.xlsx", index=False)

In [ ]:
#next step:

#https://services.gst.gov.in/services/api/search/taxpayerReturnDetails
#{"gstin":"23AAACG1653N1ZO","fy":"2026"}

#response:
# {
#     "filingStatus": [
#         [
#             {
#                 "fy": "2026-2027",
#                 "taxp": "July",
#                 "mof": "ONLINE",
#                 "dof": "10/08/2026",
#                 "rtntype": "GSTR1",
#                 "arn": "NA",
#                 "status": "Filed"
#             },
#             {
#                 "fy": "2026-2027",
#                 "taxp": "June",
#                 "mof": "ONLINE",
#                 "dof": "11/07/2026",
#                 "rtntype": "GSTR1",
#                 "arn": "NA",
#                 "status": "Filed"
#             },

In [ ]:
# POST /services/api/search/taxpayerReturnDetails HTTP/1.1
# Accept: application/json, text/plain, */*
# Accept-Encoding: gzip, deflate, br, zstd
# Accept-Language: en-US,en;q=0.9
# Connection: keep-alive
# Content-Length: 39
# Content-Type: application/json;charset=UTF-8
# Cookie: Lang=en; ak_bmsc=36A1AAA4C6E408A24A681378207EDC94~000000000000000000000000000000~YAAQtPXSF/v1OjqgAQAAugWtVgHGhVVQ5CPvmZXv87hr60FG0lI7QGDGtit1YncK/1+54m8qE8DJEcIPVl3ucE6Ng6pWB15HuZSRdd/liOOrVDYOv3vfPpxXmpUQkfonkgh8IAslkDClhuMUtNyTnSD/RaQxFXEcdWerudvvFBpRJSRiFkkcIu8NDAxwKmqU4In1xiOzUyNOEs5G6mM6MzADne6tHDtN6aRlMnRqiH2F6LflZlKBRs1J1DT0jCjS8yDcrS/ntRb+W0v69Fy5iyQohIpczdRyG97rxIPuu0/FDSx5b8k9n0h1CLzLRtVZy7x/Se0dg9L0PkerzRb084Pqnf7xQAjmUY+ZarXVb1HZBjKaejS3Bs3/IzNKHrkYjW4=; TS0134d082=010c0f54dd56551436b0cea3364b9af3729075228b76ec8c868a59a54e277b36cd95b6ecc8a5dc6158ee0aaefe0d1f8429f586c760; bm_sv=8B2B95EF60EB6B8F52DBD748B63E369B~YAAQtPXSF1myWTqgAQAAT77NVgGHXwn/7U2lzeU5cCUgzpxzp/6WwnktqhYFSZkPfo4zx4yaXb3ggV0eWSBLCd8XqcI1JCG/d1r5fwyNkRkmHQhw73J7sjukGm1xV4ZErDQh+EqR1zbZ32FIH21pvY4f/2dh+ca/vTc7qFdpANglZtDeWZ7xh1IkFtW1EKydy6AsjNxhC047ZdeFTGJ5u+xSXX/y/KtIG2glD5/+WIKdbeOC0PPd+ppYsHD+AkZg~1; TSf2521acf027=08b31d24a4ab2000e7e6c860ee90104405a0a615b044efdcad0ab660b0873a94c69a74032411ceb508a0226507113000f31a3d2f22402fb20f5f1da0d8d45d933ce81cabffeff043e3dfdd26383a9122340691d0242a4ba1eeaaa14159532fcb
# DNT: 1
# Host: services.gst.gov.in
# Origin: https://services.gst.gov.in
# Referer: https://services.gst.gov.in/services/searchtp
# Sec-Fetch-Dest: empty
# Sec-Fetch-Mode: cors
# Sec-Fetch-Site: same-origin
# User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36
# sec-ch-ua: "Chromium";v="152", "Not?A_Brand";v="24", "Google Chrome";v="152"
# sec-ch-ua-mobile: ?0
# sec-ch-ua-platform: "Windows"

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import time, json, base64, re, os
from transformers import pipeline
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pipe = pipeline("automatic-speech-recognition", model=r"D:\Developers\Kaustubh\whisper-medium")

options = webdriver.ChromeOptions()
options.set_capability("goog:loggingPrefs", {"performance": "ALL"})

headers = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json;charset=UTF-8",
    "Origin": "https://services.gst.gov.in",
    "Referer": "https://services.gst.gov.in/services/searchtp",
}


pans = [
    "AAACI4798L",
    "AAACG1653N",
    "AAACH0351E",
    "AAACV7244E",
    "AADCS1718H",
    "AAACI4341M",
    "AAACH1458C",
    "AAACR4849R",
    "AACCT8243P",
    "AACCA1963B",
]

driver = webdriver.Chrome(options=options)
# Enable DevTools Network
driver.execute_cdp_cmd("Network.enable", {})

api_url = r"https://services.gst.gov.in/services/api/search/taxpayerReturnDetails"
audio_dir = "captcha_gstn"
os.makedirs(audio_dir, exist_ok=True)
audio_dir = Path(audio_dir)


try:

    driver.get("https://services.gst.gov.in/services/searchtp")
    wait = WebDriverWait(driver, 20)
    wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, ".dimmer-holder")))
    textbox = wait.until(EC.presence_of_element_located((By.ID, "for_gstin")))
    textbox.clear()
    textbox.send_keys(f"{i}")

    # Click speaker button
    audio_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[i[contains(@class,'fa-volume-up')]]")
        )
    )

    audio_button.click()
    print("Audio button clicked...")
    time.sleep(4)
    logs = driver.get_log("performance")

    request_id = None

    for entry in logs:

        try:
            msg = json.loads(entry["message"])["message"]

            if msg["method"] == "Network.responseReceived":
                url = msg["params"]["response"]["url"]
                if "audiocaptcha" in url:

                    print("Found audio request:")
                    print(url)
                    request_id = msg["params"]["requestId"]

                    break

        except Exception:
            pass

    if not request_id:
        raise Exception("Could not find audiocaptcha request in network logs.")

    # Get response body
    body = driver.execute_cdp_cmd(
        "Network.getResponseBody", {"requestId": request_id}
    )

    if body.get("base64Encoded"):
        audio_bytes = base64.b64decode(body["body"])

    else:
        audio_bytes = body["body"].encode()

    # Save audio
    audio_path = audio_dir / f"captcha_audio_{i}.wav"

    with open(audio_path, "wb") as f:
        f.write(audio_bytes)

    print("Saved: captcha_audio.wav")
    result = pipe(audio_path)

    result = re.sub(r"[^0-9]", "", result["text"])
    print(f"DETECTED AUDIO: {result}")



    session = requests.Session()

    for cookie in driver.get_cookies():
        session.cookies.set(cookie["name"], cookie["value"])

    print(session.cookies.get_dict())

    headers = {
        "Accept": "application/json, text/plain, */*",
        "Content-Type": "application/json;charset=UTF-8",
        "Origin": "https://services.gst.gov.in",
        "Referer": "https://services.gst.gov.in/services/searchtpbypan",
    }

    payload = {"panNO": i, "captcha": result}

    response = session.post(api_url, json=payload, headers=headers, verify=False)

    print("Status:", response.status_code)
    # print("Response:")
    # print(response.text)

    if response.ok:
        print("SUCCESS !!!")
        data = response.json()
        print(response.headers.get("Content-Type"))
        print(data)

    enter_c = wait.until(EC.presence_of_element_located((By.ID, "for_gstin")))

    enter_c.clear()
    enter_c.send_keys(str(result))
    search_button = wait.until(EC.element_to_be_clickable((By.ID, "lotsearch")))
    search_button.click()

    refresh_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[i[contains(@class,'fa-refresh')]]")
        )
    )

    refresh_button.click()

except Exception as e:
    print("ERROR:", e)

finally:

    driver.quit()


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests
import time, json, base64, re, os
from transformers import pipeline
import warnings

from utils import Helper
from logger import setup_logger

warnings.filterwarnings("ignore")

pipe = pipeline(
    "automatic-speech-recognition", model=r"D:\Developers\Kaustubh\whisper-medium"
)

utils = Helper()

#GSTN SEARCH
BASE_SITE = r"https://services.gst.gov.in/services/searchtp"

# Chrome options
options = webdriver.ChromeOptions()
options.set_capability("goog:loggingPrefs", {"performance": "ALL"})

headers = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json;charset=UTF-8",
    "Origin": "https://services.gst.gov.in",
    "Referer": BASE_SITE,
}

gstins = [
    "23AAACG1653N1ZO",
    "27AAACG1653N1ZG",
    "33AAACH0351E1ZC",
    "37AAACH0351E1CF",
    "29AAACH0351E1CC",
    "02AAACH0351E2ZG",
    "07AAACH0351E1CI",
    "36AAACH0351E1CH",
    "05AAACH0351E1ZB",
    "21AAACH0351E1ZH",
    "20AAACH0351E1ZJ",
    "33AAACH0351E1CN",
    "03AAACH0351E1ZF",
    "01AAACH0351E1ZJ",
    "24AAACH0351E1CM",
    "02AAACH0351E3ZF",
    "32AAACH0351E1CP",
    "05AAACH0351E2ZA",
    "06AAACH0351E1CK",
    "18AAACH0351E1Z4",
    "07AAACH0351E1Z7",
    "08AAACH0351E1CG",
    "27AAACH0351E1Z5",
    "10AAACH0351E1CV",
    "19AAACH0351E1Z2",
    "18AAACH0351E2Z3",
    "20AAACH0351E1CU",
    "19AAACH0351E1CD",
    "23AAACH0351E1ZD",
    "18AAACH0351E4Z1",
    "18AAACH0351E1CF",
    "37AAACH0351E1Z4",
    "07AAACH0351E2Z6",
    "06AAACH0351E1Z9",
    "36AAACH0351E1Z6",
    "09AAACH0351E2Z2",
    "09AAACH0351E1CE",
    "30AAACH0351E1ZI",
    "22AAACH0351E1ZF",
    "04AAACH0351E1ZD",
    "29AAACH0351E2Z0",
    "08AAACH0351E1Z5",
    "24AAACH0351E1ZB",
    "27AAACH0351E1CG",
    "21AAACH0351E1CS",
    "03AAACH0351E1CQ",
    "01AAACH0351E1CU",
    "02AAACH0351E1ZH",
    "22AAACH0351E1CQ",
]


logger = setup_logger(name = "gstn_log")


# Map words to digits
WORD_TO_DIGIT = {
    "zero": "0",
    "oh": "0",   # sometimes Whisper says "oh" for 0
    "one": "1",
    "two": "2",
    "three": "3",
    "four": "4",
    "five": "5",
    "six": "6",
    "seven": "7",
    "eight": "8",
    "nine": "9",
}

def normalize_digits(text: str) -> str:
    # Lowercase and split
    tokens = re.findall(r"\w+", text.lower())
    digits = []
    for tok in tokens:
        if tok.isdigit():
            digits.append(tok)
        elif tok in WORD_TO_DIGIT:
            digits.append(WORD_TO_DIGIT[tok])
    return "".join(digits)


driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Network.enable", {})
api_url = r"https://services.gst.gov.in/services/api/search/taxpayerReturnDetails"
payload = {"gstin": "", "captcha": "2026"}


audio_dir = "output/gstn_audio"
os.makedirs(audio_dir, exist_ok=True)
audio_dir = Path(audio_dir)


# Create output folder
output_dir = "output/gstn_json"
os.makedirs(output_dir, exist_ok=True)


try:
    driver.get(BASE_SITE)
    wait = WebDriverWait(driver, 20)
    wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, ".dimmer-holder")))
    textbox = wait.until(EC.presence_of_element_located((By.ID, "for_gstin")))

    for i in gstins:
        textbox.clear()
        textbox.send_keys(f"{i}")

        # Click speaker button
        audio_button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[i[contains(@class,'fa-volume-up')]]")
            )
        )
        audio_button.click()
        logger.info("Audio button clicked...")

        time.sleep(5)
        logs = driver.get_log("performance")
        request_id = None

        for entry in logs:
            try:
                msg = json.loads(entry["message"])["message"]
                if msg["method"] == "Network.responseReceived":
                    url = msg["params"]["response"]["url"]
                    if "audiocaptcha" in url:
                        print("Found audio request:", url)
                        request_id = msg["params"]["requestId"]
                        break
            except Exception:
                pass

        if not request_id:
            raise Exception("Could not find audiocaptcha request in network logs.")

        # Get response body
        body = driver.execute_cdp_cmd(
            "Network.getResponseBody", {"requestId": request_id}
        )
        if body.get("base64Encoded"):
            audio_bytes = base64.b64decode(body["body"])
        else:
            audio_bytes = body["body"].encode()

        audio_path = audio_dir / f"captcha_audio_{i}.wav"
        with open(audio_path, "wb") as f:
            f.write(audio_bytes)
        logger.info(f"Saved audio: {audio_path}")

        result = pipe(audio_path)
        logger.info("Whisper raw:", result)
        captcha_digits = normalize_digits(result["text"])
        result = re.sub(r"[^0-9]", "", captcha_digits)
        
        logger.info("Normalized digits:", captcha_digits)
        print(f"DETECTED AUDIO: {result}")

        session = requests.Session()
        for cookie in driver.get_cookies():
            session.cookies.set(cookie["name"], cookie["value"])

        payload["gstin"] = i
        response = session.post(api_url, json=payload, headers=headers, verify=False)
        filename = os.path.join(output_dir, f"{i}.json")
        if response.ok:
            logger.info("SUCCESS !!!")
            data = response.json()

            # Save JSON per PAN
           
            logger.info(f"Saved JSON for {i} -> {filename}")
        else:
            logger.info(f"Failed for {i}: {response.status_code}")
            utils.save_json({"status":"Failed"}, filename)

        # Enter captcha and refresh
        enter_c = wait.until(EC.presence_of_element_located((By.ID, "for_gstin")))
        enter_c.clear()
        enter_c.send_keys(str(result))
        search_button = wait.until(EC.element_to_be_clickable((By.ID, "lotsearch")))
        search_button.click()
        refresh_button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[i[contains(@class,'fa-refresh')]]")
            )
        )
        refresh_button.click()

except Exception as e:
    logger.exception("ERROR:", e)

finally:
    driver.quit()
